In [3]:
from trot.config import configure_once

configure_once()

In [4]:
import jax.numpy as jnp
import numpy as np
from pyscf import gto
from pyscf import ao2mo, scf

L = 100
U = 8
t = 1.0
n = L 
mol = gto.M(verbose=4)
mol.incore_anyway = True
mol.nelec = (L//2,L//2)
inital_state = np.zeros((2,L,L))
for i in range(L//2):
    inital_state[0,2*i,2*i] = 1
    inital_state[1,2*i+1,2*i+1] = 1
print(inital_state)
h1 = np.zeros((n,n))
for i in range(n-1):
    h1[i,i+1] = h1[i+1,i] = -t
eri = np.zeros((n,n,n,n))
for i in range(n):
    eri[i,i,i,i] = U
mf = scf.UHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(L)
mf._eri = ao2mo.restore(8, eri, L)
mf.kernel(inital_state)


System: uname_result(system='Darwin', node='WC292CX16G', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')  Threads 1
Python 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
numpy 2.5.3  scipy 1.18.1  h5py 3.16.0
Date: Sun Oct  4 19:51:23 2026
PySCF version 2.14.0
PySCF path  /Users/fnappi/.trot/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 0
[INPUT] num. electrons = 0
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom

nuclear repulsion = 0
number of shells = 0
number of NR pGTOs = 0
number of NR cGTOs = 0
basis = sto-3g
ecp = {}
CPU time:         2.51
[[[1. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0

np.float64(-24.367343665804942)

In [ ]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc_slow import make_prop_ops
from trot.prop import blocks
import jax.numpy as jnp
import numpy as np
from pyscf import gto
from pyscf import ao2mo, scf

L = 100
U = 8
t = 1.0
n = L 
mol = gto.M(verbose=4)
mol.incore_anyway = True
mol.nelec = (L//2,L//2)
inital_state = np.zeros((2,L,L))
for i in range(L//2):
    inital_state[0,2*i,2*i] = 1
    inital_state[1,2*i+1,2*i+1] = 1
print(inital_state)
h1 = np.zeros((n,n))
for i in range(n-1):
    h1[i,i+1] = h1[i+1,i] = -t
eri = np.zeros((n,n,n,n))
for i in range(n):
    eri[i,i,i,i] = U
mf = scf.UHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(L)
mf._eri = ao2mo.restore(8, eri, L)
mf.kernel(inital_state)

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
orbitals = mf.mo_coeff
nocc = L//2
trial_data = np.block([
    [orbitals[0][:, :L//2],        np.zeros((L, L//2))],
    [np.zeros((L, L//2)),          orbitals[1][:, :L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted')
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

System: uname_result(system='Darwin', node='WC292CX16G', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')  Threads 1
Python 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
numpy 2.5.3  scipy 1.18.1  h5py 3.16.0
Date: Sun Oct  4 19:54:46 2026
PySCF version 2.14.0
PySCF path  /Users/fnappi/.trot/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 0
[INPUT] num. electrons = 0
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom

nuclear repulsion = 0
number of shells = 0
number of NR pGTOs = 0
number of NR cGTOs = 0
basis = sto-3g
ecp = {}
CPU time:        91.56
[[[1. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0

In [ ]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc_slow import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
_,orbitals = np.linalg.eig(h1)
orbitals = orbitals.real
trial_data = np.block([
    [orbitals[:, :L//2],        np.zeros((L, L//2))],
    [np.zeros((L, L//2)),          orbitals[:, :L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted')
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   14.4824590337  2.000000e+02           0       0.0
[eql   20/100]   -1.3817014010  2.890460e+02           0       2.0
[eql   40/100]   -1.6416588361  2.088443e+02           0       4.0
[eql   60/100]   -1.6655102216  2.014219e+02           0       5.7
[eql   80/100]   -1.6691246254  1.999503e+02           0       7.4
[eql  100/100]   -1.6939906193  1.997270e+02           0       9.0

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.6507990442   2.650e-02     -1.6507990442  2.000594e+02           0      0.090      13.6
[blk  100/500]   -1.6268951164   2.462e-02     -1.6030164207  2.002708e+02           0      0.082      17.7
[blk  150/500]   -1.6394515370   2.136e-02     -1.6645687184  2.00130

In [ ]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc import make_prop_ops
from trot.prop import blocks

sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=200,n_blocks=500,n_eql_blocks=100,seed=2)
ham = HamHubbard(jnp.array(h1),U)
# _,orbitals = np.linalg.eig(h1)
orbitals = mf.mo_coeff
trial_data = np.block([
    [ orbitals[0,:,:L//2],       orbitals[1,:,:L//2]],
    [ orbitals[0,:,:L//2],          -orbitals[1,:,:L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted',trial_ops)
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/100]   -1.7407111942  2.000000e+02           0       0.0
[eql   20/100]   -1.6601007765  1.996245e+02           0       1.2
[eql   40/100]   -1.6794870133  2.000711e+02           0       2.5
[eql   60/100]   -1.6433076115  1.998746e+02           0       3.3
[eql   80/100]   -1.6483990975  2.001040e+02           0       4.0
[eql  100/100]   -1.7036574511  2.001339e+02           0       4.8

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   50/500]   -1.6678856668   7.849e-03     -1.6678856668  1.999274e+02           0      0.048       7.3
[blk  100/500]   -1.6563631589   6.841e-03     -1.6448438255  1.999825e+02           0      0.040       9.3
[blk  150/500]   -1.6546708803   4.946e-03     -1.6512874176  2.00019